# MOIRAI1 — 자동 실행 큐 (고정 본 실험)

실행 순서·상태 표시·백업만 자동화합니다. 모델·checkpoint·precision·step·test window·metric은 고정 실행 commit의 코드 그대로입니다. **현재 다른 노트북에서 같은 조건이 실행 중이면 동시에 시작하지 마세요.** 모델별 별도 런타임을 사용합니다.

1. 설정 셀의 DATASETS/HORIZONS를 선택합니다. 기본 ETT 4종 × H96 × seed 3개 = 108조건. `DATASETS = ["ALL"]`은 승인된 9종, `HORIZONS = [96, 192, 336, 720]`은 네 horizon입니다. SEEDS는 기본 3개를 유지하세요.

2. 위에서부터 plan까지 준비합니다. Drive 인증은 사용자가 수행합니다. 데이터 준비는 큐가 데이터셋당 한 번 처리합니다.

3. **run 셀 한 번 클릭하면 선택한 최종 test 조건이 실행됩니다.** 추가 입력 없음. 완료 결과는 기존 runner의 identity 검증을 거쳐 건너뜁니다. 매 30초 process active는 프로세스 생존 신호이며 학습 step 진행 보장이 아닙니다. 마지막 로그가 test windows라면 평가 진행 상황입니다.

4. 조건 종료마다 소용량 ZIP/해시와 큐 상태를 Drive에 저장합니다. 마지막 export 셀은 선택 범위 전체의 결과를 ZIP 하나로 다운로드합니다. 학습 checkpoint와 test-progress는 기존 Drive 경로에 남습니다.

**중단/오류:** OOM, dependency, stale lock, 결과 불일치 시 큐 전체 중지. 자동 precision 변경·채널 축소·lock 삭제 없음. 실패 증거 보존. 원인을 해결한 후 같은 설정·commit으로 재개합니다. 새 런타임에서는 준비 셀부터 실행하세요. 실행 중 settings/plan을 바꾸지 마세요.

**시간 한도:** 18시간은 runtime 확인 셀을 처음 실행한 때부터 계산하는 사용자 지정 한도입니다. 종료 30분 전부터 다음 조건을 시작하지 않습니다. Colab의 실제 종료 시간을 알거나 보장하지 않습니다. 진행 중 조건은 강제로 끊지 않으므로 한도를 넘을 수 있습니다. 런타임 강제 종료 시 마지막 checkpoint 이후 작업은 재실행될 수 있으며, 최종 ZIP 저장도 보장되지 않습니다. Drive 원본 기록을 보존하세요.

**하드웨어:** Python 3.12 + 호환 CUDA GPU. MOIRAI Traffic H720 fine-tuning의 기존 >=75 GiB gate 유지. 40GB에서 자동 우회하지 않습니다. 새 큐 자체의 실제 Colab/GPU 실행은 아직 미검증입니다.


In [ ]:
# @title 1. 실행 범위 설정 (실행 중 변경하지 마세요)
FAMILY = "moirai1"
COMMIT = "3e18305032484239966026f14395b903c9abf268"  # 기존 결과와 동일하게 유지
DATASETS = ["ETTh1", "ETTh2", "ETTm1", "ETTm2"]  # 또는 ["ALL"]
HORIZONS = [96]  # 예: [96, 192, 336, 720]
SEEDS = [1729, 2718, 31415]
SESSION_WALL_HOURS = 18
RESERVE_MINUTES = 30
MAX_NEW_CONDITIONS = 972  # 모델당 전체 승인 조건 수; 학습 step 제한이 아님

In [ ]:
# Notebook-embedded orchestration only; the pinned main-study engine is unchanged.
# Standard library only. Embedded verbatim into notebooks 42/43 so they can run
# the historical execution commit without upgrading its source.

import hashlib
import json
import os
import signal
import subprocess
import time
import uuid
import zipfile
from datetime import UTC, datetime
from pathlib import Path


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atomic_queue_json(path, record):
    """Replace only this invocation's queue journal, never experiment evidence."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".{uuid.uuid4().hex}.tmp")
    with temporary.open("x", encoding="utf-8") as handle:
        json.dump(record, handle, ensure_ascii=False, indent=2, allow_nan=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def select_queue(plan, family, datasets, horizons, seeds):
    rows = plan["conditions"]
    available = list(dict.fromkeys(r["dataset"] for r in rows if r["family"] == family))
    if datasets == ["ALL"]:
        datasets = available
    for values, allowed, label in (
        (datasets, available, "datasets"),
        (horizons, plan["config"]["horizons"], "horizons"),
        (seeds, plan["config"]["seeds"], "seeds"),
    ):
        if not values or len(set(values)) != len(values) or not set(values) <= set(allowed):
            raise ValueError(f"Invalid {label}; allowed: {allowed}")
    selected = [
        r
        for r in rows
        if r["family"] == family
        and r["dataset"] in datasets
        and r["horizon"] in horizons
        and r["seed"] in seeds
    ]
    rates = [0.0, *plan["config"]["sampling_rates"]]
    expected = {(d, h, s, r) for d in datasets for h in horizons for s in seeds for r in rates}
    actual = {(r["dataset"], r["horizon"], r["seed"], r["rate"]) for r in selected}
    if actual != expected or len(selected) != len(expected):
        raise ValueError("Incomplete or duplicate approved grid")
    if len({r["id"] for r in selected}) != len(selected):
        raise ValueError("Duplicate condition IDs")
    for row in selected:
        name = row["id"]
        if not name or name in (".", "..") or any(c in name for c in "/\\:"):
            raise ValueError("Unsafe condition ID")
    return sorted(
        selected,
        key=lambda r: (
            datasets.index(r["dataset"]),
            horizons.index(r["horizon"]),
            seeds.index(r["seed"]),
            r["rate"],
        ),
    )


def evidence_files(out, rows):
    """Small evidence only; no optimizer checkpoints, samples, raw data or caches."""
    yield Path(out) / "plan.json"
    allowed = {"result.json", "runtime.json", "provenance.json", "selection.json", "started.json"}
    for row in rows:
        for path in sorted((Path(out) / row["id"]).glob("*.json")):
            if path.name in allowed or path.name.startswith(("failure-", "pending-")):
                yield path


def export_evidence(out, rows, destination, label):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / f"{label}-{uuid.uuid4().hex}.zip"
    temporary = target.with_suffix(".zip.partial")
    with zipfile.ZipFile(temporary, "x", zipfile.ZIP_DEFLATED) as archive:
        for path in evidence_files(out, rows):
            if path.is_symlink() or path.stat().st_size > 2_000_000:
                raise ValueError(f"Unexpected evidence file: {path.name}; original retained")
            archive.write(path, path.relative_to(out).as_posix())
    with zipfile.ZipFile(temporary) as archive:
        if archive.testzip() is not None:
            raise ValueError("ZIP integrity failure; partial retained")
    os.replace(temporary, target)
    digest = hashlib.sha256()
    with target.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    atomic_queue_json(
        target.with_suffix(".sha256.json"),
        {
            "file": target.name,
            "sha256": digest.hexdigest(),
            "bytes": target.stat().st_size,
            "condition_ids": [r["id"] for r in rows],
        },
    )
    return target


def run_process(command, cwd, log_path, label, heartbeat_seconds=30):
    """Show liveness/time, not fabricated training percentage or ETA."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    with log_path.open("x", encoding="utf-8") as log:
        child = subprocess.Popen(command, cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    code = child.wait(timeout=heartbeat_seconds)
                    break
                except subprocess.TimeoutExpired:
                    print(
                        f"  {label}: process active, elapsed {time.monotonic() - started:.0f}s",
                        flush=True,
                    )
                    with log_path.open("rb") as tail:
                        tail.seek(max(0, log_path.stat().st_size - 600))
                        text = tail.read().decode("utf-8", errors="replace").strip()
                    if text:
                        print("  latest log:", text[-400:], flush=True)
        except BaseException:
            # Stop only the subprocess this cell created. Never remove its lock/checkpoints.
            child.send_signal(signal.SIGINT)
            try:
                child.wait(timeout=30)
            except subprocess.TimeoutExpired:
                child.terminate()
                try:
                    child.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    child.kill()
                    child.wait()
            raise
    if code:
        with log_path.open("rb") as tail:
            tail.seek(max(0, log_path.stat().st_size - 4000))
            print(tail.read().decode("utf-8", errors="replace"), flush=True)
        raise subprocess.CalledProcessError(code, command)


def execute_queue(
    *,
    jobs,
    family,
    commit,
    root,
    out,
    python,
    backup_dir,
    local_logs,
    session_started,
    wall_hours=18,
    reserve_minutes=30,
    max_new=108,
    process=run_process,
    now=time.monotonic,
):
    """Sequential queue around the original one-condition CLI. Fail closed."""
    if not jobs or wall_hours <= 0 or not 0 <= reserve_minutes < wall_hours * 60 or max_new < 1:
        raise ValueError("Invalid queue/session limits")
    out, root = Path(out), Path(root)
    plan = read_json(out / "plan.json")
    if plan["commit"] != commit or any(
        r["family"] != family or r not in plan["conditions"] for r in jobs
    ):
        raise ValueError("Queue differs from pinned plan/family/commit")
    if len({r["id"] for r in jobs}) != len(jobs):
        raise ValueError("Duplicate jobs")
    run_id = uuid.uuid4().hex
    journal = Path(backup_dir) / f"queue-{run_id}.json"
    local_logs = Path(local_logs) / run_id
    record = dict(
        schema_version=1,
        wrapper_version="queue-v1",
        commit=commit,
        family=family,
        created_at=datetime.now(UTC).isoformat(),
        status="running",
        total=len(jobs),
        condition_ids=[r["id"] for r in jobs],
        verified=0,
        newly_executed=0,
        events=[],
    )
    prepared = set()
    base = [
        str(python),
        "-u",
        "-m",
        "tsfm_crossover.experiments.main_study",
        "--output",
        str(out),
        "--expected-commit",
        commit,
    ]
    atomic_queue_json(journal, record)

    def budget_reached():
        return (
            now() - session_started >= wall_hours * 3600 - reserve_minutes * 60
            or record["newly_executed"] >= max_new
        )

    try:
        for index, row in enumerate(jobs, 1):
            result = out / row["id"] / "result.json"
            existing = result.exists() and read_json(result).get("status") == "completed"
            # No next process after cutoff; completed records use the original validator too.
            if budget_reached():
                record["status"] = "session_budget_reached"
                print("Session cutoff: no next condition. Resume with the same configuration.")
                break
            started = now()
            event = {"id": row["id"], "status": "running", "existing_candidate": existing}
            record["events"].append(event)
            atomic_queue_json(journal, record)
            print(
                f"[{index}/{len(jobs)}] {'VERIFY' if existing else 'RUN'} {row['id']}", flush=True
            )
            try:
                if not existing and row["dataset"] not in prepared:
                    process(
                        [
                            str(python),
                            "-u",
                            "scripts/prepare_study_data.py",
                            "--dataset",
                            row["dataset"],
                        ],
                        root,
                        local_logs / f"prepare-{row['dataset']}.log",
                        "data integrity",
                    )
                    prepared.add(row["dataset"])
                    if budget_reached():
                        event["status"] = "not_started_session_budget"
                        record["status"] = "session_budget_reached"
                        break
                process(
                    base + ["--condition-id", row["id"]],
                    root,
                    local_logs / f"{row['id']}.log",
                    row["id"],
                )
                if not result.exists():
                    raise RuntimeError(
                        "No completed result: inspect pending resource evidence; queue stopped"
                    )
                value = read_json(result)
                if (
                    value.get("status") != "completed"
                    or value.get("identity", {}).get("commit") != commit
                    or value.get("identity", {}).get("condition") != row
                ):
                    raise ValueError("Completed result identity/status mismatch")
                event["status"] = "verified_existing" if existing else "completed"
                record["verified"] += 1
                record["newly_executed"] += not existing
            except BaseException as exc:
                event["status"] = "interrupted" if isinstance(exc, KeyboardInterrupt) else "failed"
                event["error_type"] = type(exc).__name__
                event["error"] = str(exc)[:600]
                raise
            finally:
                event["wall_seconds"] = now() - started
                # Save state first. A backup error stops this queue but never overwrites evidence.
                atomic_queue_json(journal, record)
                archive = export_evidence(out, [row], backup_dir, row["id"])
                event["backup"] = archive.name
                atomic_queue_json(journal, record)
            print(
                f"  {event['status']} ({event['wall_seconds']:.1f}s); Drive ZIP: {archive.name}",
                flush=True,
            )
        else:
            record["status"] = "completed"
    except BaseException as exc:
        record["status"] = "interrupted" if isinstance(exc, KeyboardInterrupt) else "failed"
        record["error_type"] = type(exc).__name__
        record["error"] = str(exc)[:600]
        raise
    finally:
        record["updated_at"] = datetime.now(UTC).isoformat()
        atomic_queue_json(journal, record)
        print(
            f"Queue {record['status']}: verified {record['verified']}/{len(jobs)}; "
            f"journal: {journal}",
            flush=True,
        )
    return record

In [ ]:
# @title 2. 런타임 확인
import sys

assert FAMILY == "moirai1", "Use the other model's notebook in a fresh session"
assert sys.version_info[:2] == (3, 12), "Select a Python 3.12 Colab runtime"
assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT)
if "SESSION_STARTED" not in globals():
    SESSION_STARTED = time.monotonic()
subprocess.run(["nvidia-smi"], check=True)
print("Kernel:", sys.executable, sys.version)
print("Model / immutable engine commit:", FAMILY, COMMIT)

In [ ]:
ROOT = Path("/content") / ("tsfm-main-" + COMMIT)
URL = "https://github.com/Han-Youseung/tsfm-zero-few-shot-crossover.git"
if not ROOT.exists():
    subprocess.run(["git", "clone", URL, str(ROOT)], check=True)
assert (
    subprocess.check_output(
        ["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True
    ).strip()
    == URL
)
assert not subprocess.check_output(
    ["git", "-C", str(ROOT), "status", "--porcelain", "--untracked-files=no"], text=True
).strip()
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
os.chdir(ROOT)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip() == COMMIT
print("Immutable execution commit:", COMMIT)

In [ ]:
from google.colab import drive

# User executes this authorization cell. No credentials are stored in source/output.
drive.mount("/content/drive")
PERSIST = Path("/content/drive/MyDrive/tsfm-main-study")
OUT = PERSIST / COMMIT / FAMILY
OUT.mkdir(parents=True, exist_ok=True)
# Persist public data only. HF/Xet symlink caches must stay on local disk.
for relative in ("data/source_cache",):
    target = PERSIST / "public-cache" / relative
    target.mkdir(parents=True, exist_ok=True)
    link = ROOT / relative
    link.parent.mkdir(parents=True, exist_ok=True)
    if not link.exists():
        link.symlink_to(target, target_is_directory=True)
# Local model cache guard: never automatically remove an existing Drive cache.
model_cache = ROOT / ".cache"
drive_root = Path("/content/drive").resolve()
for candidate in (model_cache, model_cache / FAMILY, model_cache / "hf-home", model_cache / "xet"):
    resolved = candidate.resolve()
    assert resolved != drive_root and drive_root not in resolved.parents, (
        "Model cache points to Drive; preserve it and review recovery before continuing."
    )
model_cache.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(model_cache / "hf-home")
os.environ["HF_XET_CACHE"] = str(model_cache / "xet")
print("Local model cache (re-download after runtime loss):", model_cache.resolve())
print("Drive results/checkpoints:", OUT)
print("Temporary /content is not persistent. Do not remove previous failure evidence.")

In [ ]:
ENV = Path("/content") / ("venv-main-" + FAMILY)
PY = ENV / "bin/python"
if not PY.exists():
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "python3.12-venv"], check=True)
    subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)
subprocess.run(
    [str(PY), "-m", "pip", "install", "-r", f"requirements/{FAMILY}-main.txt"], check=True
)
subprocess.run([str(PY), "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run([str(PY), "-m", "pip", "check"], check=True)
probe = """
import json, sys, torch
assert torch.cuda.is_available()
assert sys.version_info[:2] == (3, 12)
print(json.dumps(dict(interpreter=sys.executable, python=sys.version, torch=torch.__version__,
    cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(),
    vram_gib=torch.cuda.get_device_properties(0).total_memory/2**30)))
"""
subprocess.run([str(PY), "-c", probe], check=True)
print("Every probe/training/data subprocess uses", PY, "; kernel imports no vendor package.")

In [ ]:
# @title 3. 승인된 계획 선택 (학습 실행 아님)
BASE = [
    str(PY),
    "-u",
    "-m",
    "tsfm_crossover.experiments.main_study",
    "--output",
    str(OUT),
    "--expected-commit",
    COMMIT,
]
subprocess.run(BASE, check=True)
plan = read_json(OUT / "plan.json")
assert len(plan["conditions"]) == 1944
jobs = select_queue(plan, FAMILY, DATASETS, HORIZONS, SEEDS)
QUEUE_SETTINGS = (FAMILY, COMMIT, tuple(DATASETS), tuple(HORIZONS), tuple(SEEDS))
BACKUPS = PERSIST / "queue-exports" / COMMIT / FAMILY
existing = sum((OUT / r["id"] / "result.json").exists() for r in jobs)
print("Selected conditions:", len(jobs))
print("Existing result files (not yet validated):", existing)
print("Potential unfinished conditions:", len(jobs) - existing)
print("First:", jobs[0]["id"], "| Last:", jobs[-1]["id"])
print("Drive automatic backups:", BACKUPS)
print("No final test score is used for ordering, settings, or selection.")

In [ ]:
# @title 4. 모델 실행 — 한 번 클릭하면 선택한 큐 순차 실행
assert (FAMILY, COMMIT, tuple(DATASETS), tuple(HORIZONS), tuple(SEEDS)) == QUEUE_SETTINGS, (
    "Settings changed: rerun plan before starting the queue."
)
assert jobs == select_queue(read_json(OUT / "plan.json"), FAMILY, DATASETS, HORIZONS, SEEDS)
queue_result = execute_queue(
    jobs=jobs,
    family=FAMILY,
    commit=COMMIT,
    root=ROOT,
    out=OUT,
    python=PY,
    backup_dir=BACKUPS,
    local_logs=Path("/content/tsfm-queue-logs"),
    session_started=SESSION_STARTED,
    wall_hours=SESSION_WALL_HOURS,
    reserve_minutes=RESERVE_MINUTES,
    max_new=MAX_NEW_CONDITIONS,
)

In [ ]:
# @title 5. 선택 범위 전체 결과 ZIP 다운로드 (부분 완료여도 가능)
from google.colab import files

archive = export_evidence(OUT, jobs, PERSIST / "exports", f"{FAMILY}-main-queue")
print("Drive backup:", archive)
print("SHA256 metadata:", archive.with_suffix(".sha256.json"))
print("Partial results are allowed; ZIP creation does not mean all conditions completed.")
files.download(str(archive))

## 다음 실행 / 결과 회수

큐 상태 JSON과 조건별 ZIP: `MyDrive/tsfm-main-study/queue-exports/<실행 commit>/moirai1/`. 전체 선택 범위 ZIP: `MyDrive/tsfm-main-study/exports/`. 다운로드가 차단되면 Drive에서 직접 받으세요. ZIP은 기존 조건별 디렉터리와 plan.json을 유지합니다. 로컬 provenance 검증 후 분석에 사용하세요. ZIP 생성은 전체 실험 완료를 의미하지 않습니다.

진행률은 이번 큐에서 검증 완료한 조건 수 / 선택한 전체 조건 수입니다. 학습 완료 ETA는 제공하지 않습니다. 완료된 조건도 기존 runner로 검증하며 새로 학습하지 않습니다. 큐 시간 한도가 끝난 경우 새 런타임에서 이어가세요. TTM과 MOIRAI를 같은 환경에 혼합 설치하지 마세요.
